# EDA for Datacenter GPU Consumption
This notebook analyzes consumption trends based on historical data segmented by Datacenter and Customer Type.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# Sample data
data = {
    "Metric": ["Instance GPU"] * 12,
    "Select Stack By": ["ATLANTA (Atlanta, GA, USA)"] * 6 + ["LONDON (London, UK)"] * 6,
    "Pivot By": ["Internal"] * 3 + ["External"] * 3 + ["Internal"] * 3 + ["External"] * 3,
    "MonthStart": ["01-01-2024", "01-02-2024", "01-03-2024", "01-04-2024", "01-05-2024", "01-06-2024",
                   "01-01-2024", "01-02-2024", "01-03-2024", "01-04-2024", "01-05-2024", "01-06-2024"],
    "Percentile95": [5, 6, 6, 7, 8, 9, 10, 12, 13, 15, 16, 18]
}

# Create DataFrame
df = pd.DataFrame(data)

# Rename columns
df.rename(columns={
    "Select Stack By": "Datacenter",
    "Pivot By": "Customer Type",
    "MonthStart": "Month",
    "Percentile95": "Consumption"
}, inplace=True)

# Parse date
df["Month"] = pd.to_datetime(df["Month"], format="%d-%m-%Y")
df.head()

In [ ]:
# Group and calculate rolling metrics
grouped = df.groupby(["Month", "Datacenter", "Customer Type"])["Consumption"].sum().reset_index()
grouped["RollingMean"] = grouped.groupby(["Datacenter", "Customer Type"])["Consumption"].transform(lambda x: x.rolling(2).mean())
grouped["RollingStd"] = grouped.groupby(["Datacenter", "Customer Type"])["Consumption"].transform(lambda x: x.rolling(2).std())
grouped.head()

In [ ]:
# Plot trends by segment
plt.figure(figsize=(12, 6))
for key, grp in grouped.groupby(["Datacenter", "Customer Type"]):
    plt.plot(grp["Month"], grp["Consumption"], marker='o', label=f"{key[0]} - {key[1]}")
plt.title("Consumption Trends by Datacenter and Customer Type")
plt.xlabel("Month")
plt.ylabel("Consumption (Percentile95)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()